# Lab thực hành: Cây quyết định

Trong bài tập này, bạn sẽ triển khai decision tree từ đầu và áp dụng nó vào nhiệm vụ phân loại xem nấm ăn được hay nấm độc.

# phác thảo
- [ 1 - Packages ](#1)
- [ 2 -  Problem Statement](#2)
- [ 3 - Dataset](#3)
  - [ 3.1 One hot encoded dataset](#3.1)
- [ 4 - Decision Tree Refresher](#4)
  - [ 4.1  Calculate entropy](#4.1)
    - [ Exercise 1](#ex01)
  - [ 4.2  Split dataset](#4.2)
    - [ Exercise 2](#ex02)
  - [ 4.3  Calculate information gain](#4.3)
    - [ Exercise 3](#ex03)
  - [ 4.4  Get best split](#4.4)
    - [ Exercise 4](#ex04)
- [ 5 - Building the tree](#5)


<a name="1"></a>
##1 - Gói 

Trước tiên, hãy chạy ô bên dưới để nhập tất cả các gói mà bạn sẽ cần trong nhiệm vụ này.
- [numpy](https://www.numpy.org) là gói cơ bản để làm việc với ma trận trong Python.
- [matplotlib](https://matplotlib.org) là thư viện nổi tiếng để vẽ đồ thị trong Python.
- ``utils.py`` chứa các hàm trợ giúp cho bài tập này. Bạn không cần phải sửa đổi mã trong file này.


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from public_tests import *

%matplotlib inline

<a name="2"></a>
##2 - Báo cáo vấn đề

Giả sử bạn đang thành lập một công ty trồng và bán nấm dại. 
- Vì không phải tất cả các loại nấm đều ăn được nên bạn muốn biết một loại nấm nào đó có thể ăn được hay có độc dựa trên đặc tính vật lý của nó
- Bạn có một số dữ liệu hiện có mà bạn có thể sử dụng cho nhiệm vụ này. 

Bạn có thể sử dụng dữ liệu để giúp bạn xác định loại nấm nào có thể được bán một cách an toàn không? 

Lưu ý: Dữ liệu được sử dụng chỉ nhằm mục đích minh họa. Nó không nhằm mục đích hướng dẫn cách xác định các loại nấm ăn được.



<a name="3"></a>
##3 - Tập dữ liệu

Bạn sẽ bắt đầu bằng cách tải tập dữ liệu cho tác vụ này. Tập dữ liệu bạn đã thu thập như sau:

| Màu mũ | Hình Thân Cây | Đơn độc | Ăn được |
|:--------:|:-----------:|:--------:|:------:|
|   Nâu |   Giảm dần |    Có |    1 |
|   Nâu |  Phóng to |    Có |    1 |
|   Nâu |  Phóng to |    Không |    0 |
|   Nâu |  Phóng to |    Không |    0 |
|   Nâu |   Giảm dần |    Có |    1 |
|    Đỏ |   Giảm dần |    Có |    0 |
|    Đỏ |  Phóng to |    Không |    0 |
|   Nâu |  Phóng to |    Có |    1 |
|    Đỏ |   Giảm dần |    Không |    1 |
|   Nâu |  Phóng to |    Không |    0 |


- Bạn có 10 ví dụ về nấm. Với mỗi ví dụ, bạn có
    - Ba feature
        - Màu nắp (`Brown` hoặc `Red`),
        - Hình dáng cuống (`Tapering` hoặc `Enlarging`), và
        - Đơn độc (`Yes` hoặc `No`)
    - Nhãn
        - Ăn được (`1` biểu thị có hoặc `0` biểu thị độc)

<a name="3.1"></a>
### 3.1 Một tập dữ liệu được mã hóa nóng
Để dễ thực hiện, chúng ta đã mã hóa một lần các feature (biến chúng thành các feature có giá trị 0 hoặc 1)

| Mũ Nâu | Thân cây thon gọn | Đơn độc | Ăn được |
|:--------:|:--------------------:|:--------:|:------:|
|     1 |           1 |     1 |    1 |
|     1 |           0 |     1 |    1 |
|     1 |           0 |     0 |    0 |
|     1 |           0 |     0 |    0 |
|     1 |           1 |     1 |    1 |
|     0 |           1 |     1 |    0 |
|     0 |           0 |     0 |    0 |
|     1 |           0 |     1 |    1 |
|     0 |           1 |     0 |    1 |
|     1 |           0 |     0 |    0 |

Vì vậy,
- `X_train` chứa ba feature cho mỗi ví dụ 
    - Màu nâu (Giá trị `1` biểu thị màu nắp "Nâu" và `0` biểu thị màu nắp "Đỏ")
    - Hình dạng thuôn nhọn (Giá trị `1` biểu thị "Hình dạng cuống thuôn nhọn" và `0` biểu thị hình dạng cuống "Mở rộng")
    - Đơn độc (Giá trị `1` biểu thị "Có" và `0` biểu thị "Không")

- `y_train` là nấm có ăn được không 
    - `y = 1` biểu thị có thể ăn được
    - `y = 0` biểu thị chất độc


In [7]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

#### Xem các biến
Hãy làm quen nhiều hơn với tập dữ liệu của bạn.  
- Cách tốt nhất để bắt đầu là in ra từng biến và xem nó chứa gì.

Mã bên dưới in một số phần tử đầu tiên của `X_train` và loại biến.


In [8]:
print("First few elements of X_train:\n", X_train[:5])
print("Type of X_train:",type(X_train))

First few elements of X_train:
 [[1 1 1]
 [1 0 1]
 [1 0 0]
 [1 0 0]
 [1 1 1]]
Type of X_train: <class 'numpy.ndarray'>


Bây giờ, hãy làm tương tự cho `y_train`


In [9]:
print("First few elements of y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

First few elements of y_train: [1 1 0 0 1]
Type of y_train: <class 'numpy.ndarray'>


#### Kiểm tra chiều của các biến của bạn

Một cách hữu ích khác để làm quen với dữ liệu của bạn là xem kích thước của nó.

Vui lòng in hình dạng `X_train` và `y_train` và xem bạn có bao nhiêu traning example trong tập dữ liệu của mình.


In [10]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(X_train))

The shape of X_train is: (10, 3)
The shape of y_train is:  (10,)
Number of training examples (m): 10


<a name="4"></a>
##4 - Làm mới cây quyết định

Trong Lab thực hành này, bạn sẽ xây dựng decision tree dựa trên tập dữ liệu được cung cấp.

- Nhắc lại các bước xây dựng decision tree như sau:
    - Bắt đầu với tất cả các ví dụ tại nút gốc
    - Tính toán mức tăng thông tin để phân tách trên tất cả các feature có thể và chọn một feature có mức tăng thông tin cao nhất
    - Tách tập dữ liệu theo feature đã chọn và tạo các nhánh trái và phải của cây
    - Tiếp tục lặp lại quá trình phân tách cho đến khi đạt tiêu chí dừng
  
  
- Trong lab này, bạn sẽ triển khai các chức năng sau, cho phép bạn chia một nút thành các nhánh trái và phải bằng cách sử dụng feature thu được thông tin cao nhất
    - Tính entropy tại một nút 
    - Chia tập dữ liệu tại một nút thành các nhánh trái và phải dựa trên một feature nhất định
    - Tính toán mức tăng thông tin từ việc phân tách trên một feature nhất định
    - Chọn feature tối đa hóa việc thu được thông tin
    
- Sau đó, chúng ta sẽ sử dụng các chức năng trợ giúp mà bạn đã triển khai để xây dựng decision tree bằng cách lặp lại quá trình phân tách cho đến khi đáp ứng tiêu chí dừng 
    - Đối với Lab này, tiêu chí dừng mà chúng ta đã chọn là đặt độ sâu tối đa là 2


<a name="4.1"></a>
### 4.1 Tính entropy

Trước tiên, bạn sẽ viết một hàm trợ giúp có tên `compute_entropy` để tính toán entropy (thước đo độ tạp chất) tại một nút. 
- Hàm lấy mảng numpy (`y`) cho biết các ví dụ trong nút đó là ăn được (`1`) hay độc(`0`) 

Hoàn thành hàm `compute_entropy()` bên dưới để:
* Tính $p_1$, là tỷ lệ của các ví dụ có thể ăn được (tức là có giá trị = `1` trong `y`)
* Entropy sau đó được tính như sau 

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$
* Lưu ý 
    * Nhật ký được tính bằng cơ số $2$
    * Vì mục đích thực hiện, $0\text{log}_2(0) = 0$. Nghĩa là, nếu `p_1 = 0` hoặc `p_1 = 1`, hãy đặt entropy thành `0`
    * Đảm bảo kiểm tra xem dữ liệu tại một nút không trống (tức là `len(y) != 0`). Trả về `0` nếu đúng
    
<a name="ex01"></a>
### Bài tập 1

Vui lòng hoàn thành chức năng `compute_entropy()` bằng cách sử dụng các hướng dẫn trước đó.
    
Nếu gặp khó khăn, bạn có thể xem các gợi ý được trình bày sau ô bên dưới để giúp bạn thực hiện.


In [22]:
# UNQ_C1
# GRADED FUNCTION: compute_entropy

def compute_entropy(y):
    """
    Computes the entropy for 
    
    Args:
       y (ndarray): Numpy array indicating whether each example at a node is
           edible (`1`) or poisonous (`0`)
       
    Returns:
        entropy (float): Entropy at that node
        
    """
    # Bạn cần trả về chính xác các biến sau    entropy = 0.
    
    # ## BẮT ĐẦU MÃ TẠI ĐÂY ###    if len(y) != 0:
        p1 = p1 = len(y[y == 1]) / len(y) 
     # Với p1 = 0 và 1, đặt entropy thành 0 (để xử lý 0log0)        if p1 != 0 and p1 != 1:
             entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)
        else:
             entropy = 0
    # ## KẾT THÚC MÃ TẠI ĐÂY ###    
    return entropy

<details>
  <summary><font size="3" color="darkgreen"><b>Nhấp để xem gợi ý</b></font></summary>
    
    
   * Để tính `p1`
       * Bạn có thể lấy tập hợp con các ví dụ trong `y` có giá trị `1` là `y[y == 1]`
       * Bạn có thể sử dụng `len(y)` để lấy số lượng ví dụ trong `y`
   * Để tính `entropy`
       * <a href="https://numpy.org/doc/stable/reference/generated/numpy.log2.html">np.log2</a> các bạn tính logarit cơ số 2 cho mảng numpy
       * Nếu giá trị của `p1` là 0 hoặc 1, hãy đảm bảo đặt entropy thành `0` 
     
    <details>
          <summary><font size="2" color="darkblue"><b> Nhấp để biết thêm gợi ý</b></font></summary>
        
    * Đây là cách bạn có thể cấu trúc việc triển khai tổng thể cho chức năng này
    ```python 
    def compute_entropy(y):
        
        # You need to return the following variables correctly
        entropy = 0.

        ### START CODE HERE ###
        if len(y) != 0:
            # Your code here to calculate the fraction of edible examples (i.e with value = 1 in y)
            p1 =

            # For p1 = 0 and 1, set the entropy to 0 (to handle 0log0)
            if p1 != 0 and p1 != 1:
                # Your code here to calculate the entropy using the formula provided above
                entropy = 
            else:
                entropy = 0. 
        ### END CODE HERE ###        

        return entropy
    ```
    
    Nếu vẫn gặp khó khăn, bạn có thể xem các gợi ý bên dưới để tìm ra cách tính `p1` và `entropy`.
    
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý tính p1</b></font></summary>
           &emsp; &emsp; Bạn có thể tính p1 dưới dạng <code>p1 = len(y[y == 1]) / len(y) </code>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý tính entropy</b></font></summary>
          &emsp; &emsp; Bạn có thể tính entropy dưới dạng <code>entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)</code>
    </details>
        
    </details>

</details>


Bạn có thể kiểm tra xem việc triển khai của mình có đúng hay không bằng cách chạy mã kiểm tra sau:


In [23]:
# Tính toán entropy tại nút gốc (tức là với tất cả các ví dụ)# Vì chúng ta có 5 loại nấm ăn được và 5 loại nấm không ăn được nên entropy sẽ là 1"
print("Entropy at root node: ", compute_entropy(y_train)) 

# KIỂM TRA ĐƠN VỊcompute_entropy_test(compute_entropy)

Entropy at root node:  1.0
 All tests passed.


**Đầu ra dự kiến**:
<table>
  <tr>
    <td> <b>Entropy tại nút gốc:<b> 1.0 </td> 
  </tr>
</table>


<a name="4.2"></a>
### 4.2 Chia tập dữ liệu

Tiếp theo, bạn sẽ viết một hàm trợ giúp có tên `split_dataset` để nhận dữ liệu tại một nút và một feature để phân tách và chia dữ liệu đó thành các nhánh trái và phải. Sau này trong Lab, bạn sẽ triển khai mã để tính toán mức độ phân chia tốt như thế nào.

- Hàm lấy dữ liệu training, danh sách chỉ số của các điểm dữ liệu tại nút đó cùng với feature để phân chia. 
- Nó phân tách dữ liệu và trả về tập hợp con các chỉ số ở nhánh trái và nhánh phải.
- Ví dụ: giả sử chúng ta đang bắt đầu từ nút gốc (vì vậy `node_indices = [0,1,2,3,4,5,6,7,8,9]`) và chúng ta đã chọn tách theo feature `0`, tức là ví dụ có nắp màu nâu hay không.
    - Khi đó đầu ra của hàm là `left_indices = [0,1,2,3,4,7,9]` và `right_indices = [5,6,8]`
    
| Chỉ mục | Mũ Nâu | Thân cây thon gọn | Đơn độc | Ăn được |
|:------:|:----------:|:----------------------:|:--------:|:------:|
|   0 |     1 |           1 |     1 |    1 |
|   1 |     1 |           0 |     1 |    1 |
|   2 |     1 |           0 |     0 |    0 |
|   3 |     1 |           0 |     0 |    0 |
|   4 |     1 |           1 |     1 |    1 |
|   5 |     0 |           1 |     1 |    0 |
|   6 |     0 |           0 |     0 |    0 |
|   7 |     1 |           0 |     1 |    1 |
|   8 |     0 |           1 |     0 |    1 |
|   9 |     1 |           0 |     0 |    0 |

<a name="ex02"></a>
### Bài tập 2

Vui lòng hoàn thành chức năng `split_dataset()` hiển thị bên dưới

- Đối với mỗi chỉ mục trong `node_indices`
    - Nếu giá trị `X` tại chỉ mục đó cho feature đó là `1`, hãy thêm chỉ mục vào `left_indices`
    - Nếu giá trị `X` tại chỉ mục đó cho feature đó là `0`, hãy thêm chỉ mục vào `right_indices`

Nếu gặp khó khăn, bạn có thể xem các gợi ý được trình bày sau ô bên dưới để giúp bạn thực hiện.


In [24]:
# UNQ_C2
# GRADED FUNCTION: split_dataset

def split_dataset(X, node_indices, feature):
    """
    Splits the data at the given node into
    left and right branches
    
    Args:
        X (ndarray):             Data matrix of shape(n_samples, n_features)
        node_indices (list):  List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        left_indices (list): Indices with feature value == 1
        right_indices (list): Indices with feature value == 0
    """
    
    # Bạn cần trả về chính xác các biến sau    left_indices = []
    right_indices = []
    
    # ## BẮT ĐẦU MÃ TẠI ĐÂY ###    for i in node_indices:   
        if X[i][feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    # ## KẾT THÚC MÃ TẠI ĐÂY ###        
    return left_indices, right_indices

<details>
  <summary><font size="3" color="darkgreen"><b>Nhấp để xem gợi ý</b></font></summary>
    
    
   * Đây là cách bạn có thể cấu trúc việc triển khai tổng thể cho chức năng này
    ```python 
    def split_dataset(X, node_indices, feature):
    
        # You need to return the following variables correctly
        left_indices = []
        right_indices = []

        ### START CODE HERE ###
        # Go through the indices of examples at that node
        for i in node_indices:   
            if # Your code here to check if the value of X at that index for the feature is 1
                left_indices.append(i)
            else:
                right_indices.append(i)
        ### END CODE HERE ###
        
    return left_indices, right_indices
    ```
    <details>
          <summary><font size="2" color="darkblue"><b> Nhấp để biết thêm gợi ý</b></font></summary>
        
    Điều kiện là <code> nếu X[i][feature] == 1:</code>.
        
    </details>

</details>


Bây giờ, hãy kiểm tra việc triển khai của bạn bằng cách sử dụng các khối mã bên dưới. Hãy thử tách tập dữ liệu tại nút gốc, chứa tất cả các ví dụ ở feature 0 (Nắp nâu) như chúng ta đã thảo luận ở trên


In [25]:
root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Hãy thoải mái thử nghiệm với các biến này# Tập dữ liệu chỉ có ba feature nên giá trị này có thể là 0 (Nắp nâu), 1 (Hình dạng cuống thon) hoặc 2 (Solitary)feature = 0

left_indices, right_indices = split_dataset(X_train, root_indices, feature)

print("Left indices: ", left_indices)
print("Right indices: ", right_indices)

# KIỂM TRA ĐƠN VỊsplit_dataset_test(split_dataset)

Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
 All tests passed.


**Đầu ra dự kiến**:
```
Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
```


<a name="4.3"></a>
### 4.3 Tính toán mức tăng thông tin

Tiếp theo, bạn sẽ viết một hàm có tên `information_gain` để nhận dữ liệu training, các chỉ mục tại một nút và một feature để phân tách và trả về thông tin thu được từ quá trình phân tách.

<a name="ex03"></a>
### Bài tập 3

Vui lòng hoàn thành hàm `compute_information_gain()` hiển thị bên dưới để tính toán

$$\text{Information Gain} = H(p_1^\text{node})- (w^{\text{left}}H(p_1^\text{left}) + w^{\text{right}}H(p_1^\text{right}))$$

ở đâu 
- $H(p_1^\text{node})$ là entropy tại nút 
- $H(p_1^\text{left})$ và $H(p_1^\text{right})$ là entropy ở nhánh trái và nhánh phải do sự phân tách
- $w^{\text{left}}$ và $w^{\text{right}}$ lần lượt là tỷ lệ các ví dụ ở nhánh trái và nhánh phải

Lưu ý:
- Bạn có thể sử dụng hàm `compute_entropy()` mà bạn đã triển khai ở trên để tính entropy
- chúng ta đã cung cấp một số mã khởi đầu sử dụng hàm `split_dataset()` mà bạn đã triển khai ở trên để phân tách tập dữ liệu 

Nếu gặp khó khăn, bạn có thể xem các gợi ý được trình bày sau ô bên dưới để giúp bạn thực hiện.


In [26]:
# UNQ_C3
# GRADED FUNCTION: compute_information_gain

def compute_information_gain(X, y, node_indices, feature):
    
    """
    Compute the information of splitting the node on a given feature
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
   
    Returns:
        cost (float):        Cost computed
    
    """    
    # Tách tập dữ liệu    left_indices, right_indices = split_dataset(X, node_indices, feature)
    
    # Một số biến hữu ích    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    # Bạn cần trả về chính xác các biến sau    information_gain = 0
    
    # ## BẮT ĐẦU MÃ TẠI ĐÂY ###    node_entropy = compute_entropy(y_node)
    left_entropy = compute_entropy(y_left)
    right_entropy = compute_entropy(y_right)
    
    # Trọng lượng    w_left = len(X_left) / len(X_node)
    w_right = len(X_right) / len(X_node)
    
    # Entropy có trọng số    weighted_entropy = w_left * left_entropy + w_right * right_entropy
    
    # Thu được thông tin    information_gain = node_entropy - weighted_entropy
    
    # ## KẾT THÚC MÃ TẠI ĐÂY ###    
    return information_gain

<details>
  <summary><font size="3" color="darkgreen"><b>Nhấp để xem gợi ý</b></font></summary>
    
    
   * Đây là cách bạn có thể cấu trúc việc triển khai tổng thể cho chức năng này
    ```python 
    def compute_information_gain(X, y, node_indices, feature):
        # Split dataset
        left_indices, right_indices = split_dataset(X, node_indices, feature)

        # Some useful variables
        X_node, y_node = X[node_indices], y[node_indices]
        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        # You need to return the following variables correctly
        information_gain = 0

        ### START CODE HERE ###
        # Your code here to compute the entropy at the node using compute_entropy()
        node_entropy = 
        # Your code here to compute the entropy at the left branch
        left_entropy = 
        # Your code here to compute the entropy at the right branch
        right_entropy = 

        # Your code here to compute the proportion of examples at the left branch
        w_left = 
        
        # Your code here to compute the proportion of examples at the right branch
        w_right = 

        # Your code here to compute weighted entropy from the split using 
        # w_left, w_right, left_entropy and right_entropy
        weighted_entropy = 

        # Your code here to compute the information gain as the entropy at the node
        # minus the weighted entropy
        information_gain = 
        ### END CODE HERE ###  

        return information_gain
    ```
    Nếu bạn vẫn gặp khó khăn, hãy xem các gợi ý bên dưới.
    
    <details>
          <summary><font size="2" color="darkblue"><b> Gợi ý tính entropy</b></font></summary>
        
    <code>node_entropy = tính_entropy(y_node)</code><br>
    <code>left_entropy = tính_entropy(y_left)</code><br>
    <code>right_entropy = tính_entropy(y_right)</code>
        
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý tính w_left và w_right</b></font></summary>
           <code>w_left = len(X_left) / len(X_node)</code><br>
           <code>w_right = len(X_right) / len(X_node)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý tính trọng số_entropy</b></font></summary>
           <code>weighted_entropy = w_left * left_entropy + w_right * right_entropy</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý tính thông tin_gain</b></font></summary>
           <code> thông tin_gain = nút_entropy - trọng_entropy</code>
    </details>


</details>


Giờ đây, bạn có thể kiểm tra việc triển khai của mình bằng cách sử dụng ô bên dưới và tính toán thông tin thu được từ việc phân tách từng feature


In [27]:
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print("Information Gain from splitting the root on brown cap: ", info_gain0)
    
info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)

# KIỂM TRA ĐƠN VỊcompute_information_gain_test(compute_information_gain)

Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
 All tests passed.


**Đầu ra dự kiến**:
```
Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
```


Việc chia tách trên "Solitary" (feature = 2) tại nút gốc sẽ mang lại mức thu được thông tin tối đa. Vì vậy, đây là feature tốt nhất để phân chia ở nút gốc.


<a name="4.4"></a>
### 4.4 Được chia tốt nhất
Bây giờ, hãy viết một hàm để có được feature tốt nhất để phân chia bằng cách tính toán mức thu được thông tin từ mỗi feature như chúng ta đã làm ở trên và trả về feature mang lại mức thu được thông tin tối đa

<a name="ex04"></a>
### Bài tập 4
Vui lòng hoàn thành chức năng `get_best_split()` hiển thị bên dưới.
- Hàm lấy dữ liệu training cùng với các chỉ số của datapoint tại nút đó
- Đầu ra của hàm là feature mang lại mức thu được thông tin tối đa 
    - Bạn có thể sử dụng hàm `compute_information_gain()` để duyệt qua các feature và tính toán thông tin cho từng feature
Nếu gặp khó khăn, bạn có thể xem các gợi ý được trình bày sau ô bên dưới để giúp bạn thực hiện.


In [36]:
# UNQ_C4
# GRADED FUNCTION: get_best_split

def get_best_split(X, y, node_indices):   
    """
    Returns the optimal feature and threshold value
    to split the node data 
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.

    Returns:
        best_feature (int):     The index of the best feature to split
    """    
    
    # Một số biến hữu ích    num_features = X.shape[1]
    
    # Bạn cần trả về chính xác các biến sau    best_feature = -1
    
    # ## BẮT ĐẦU MÃ TẠI ĐÂY ###    max_info_gain=0
    for feature in range(num_features):
        info_gain = compute_information_gain(X, y, node_indices, feature)
        if info_gain > max_info_gain:
            max_info_gain = info_gain
            best_feature = feature
                
        
    # ## KẾT THÚC MÃ TẠI ĐÂY ##       
    return best_feature


<details>
  <summary><font size="3" color="darkgreen"><b>Nhấp để xem gợi ý</b></font></summary>
    
    
   * Đây là cách bạn có thể cấu trúc việc triển khai tổng thể cho chức năng này
    
    ```python 
    def get_best_split(X, y, node_indices):   

        # Some useful variables
        num_features = X.shape[1]

        # You need to return the following variables correctly
        best_feature = -1

        ### START CODE HERE ###
        max_info_gain = 0

        # Iterate through all features
        for feature in range(num_features): 
            
            # Your code here to compute the information gain from splitting on this feature
            info_gain = 
            
            # If the information gain is larger than the max seen so far
            if info_gain > max_info_gain:  
                # Your code here to set the max_info_gain and best_feature
                max_info_gain = 
                best_feature = 
        ### END CODE HERE ##    
   
    return best_feature
    ```
    Nếu bạn vẫn gặp khó khăn, hãy xem các gợi ý bên dưới.
    
    <details>
          <summary><font size="2" color="darkblue"><b> Gợi ý tính thông tin_gain</b></font></summary>
        
    <code>info_gain = tính_information_gain(X, y, nút_indices, feature)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>Gợi ý cập nhật max_info_gain và best_feature</b></font></summary>
           <code>max_info_gain = thông tin_gain</code><br>
           <code>best_feature = feature</code>
    </details>
</details>


Bây giờ, hãy kiểm tra việc triển khai chức năng của bạn bằng cách sử dụng ô bên dưới.


In [37]:
best_feature = get_best_split(X_train, y_train, root_indices)
print("Best feature to split on: %d" % best_feature)

# KIỂM TRA ĐƠN VỊget_best_split_test(get_best_split)

Best feature to split on: 2
 All tests passed.


Như chúng ta đã thấy ở trên, hàm trả về feature tốt nhất để phân chia ở nút gốc là feature 2 ("Solitary")


<a name="5"></a>
##5 – Xây dựng cây

Trong phần này, chúng ta sử dụng các hàm bạn đã triển khai ở trên để tạo decision tree bằng cách liên tục chọn feature tốt nhất để phân tách cho đến khi đạt tiêu chí dừng (độ sâu tối đa là 2).

Bạn không cần phải thực hiện bất cứ điều gì cho phần này.


In [38]:
# Not graded
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    """
    Build a tree using the recursive algorithm that split the dataset into 2 subgroups at each node.
    This function just prints the tree.
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        branch_name (string):   Name of the branch. ['Root', 'Left', 'Right']
        max_depth (int):        Max depth of the resulting tree. 
        current_depth (int):    Current depth. Parameter used during recursive call.
   
    """ 

    # Đã đạt đến độ sâu tối đa - dừng phân chia    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   
    # Nếu không, hãy phân chia và chia nhỏ dữ liệu một cách tốt nhất    # Nhận feature và ngưỡng tốt nhất tại nút này    best_feature = get_best_split(X, y, node_indices) 
    tree.append((current_depth, branch_name, best_feature, node_indices))
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # Chia tập dữ liệu ở feature tốt nhất    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    
    # tiếp tục chia con trái và con phải. Tăng độ sâu hiện tại    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)


In [39]:
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)

 Depth 0, Root: Split on feature: 2
- Depth 1, Left: Split on feature: 0
  -- Left leaf node with indices [0, 1, 4, 7]
  -- Right leaf node with indices [5]
- Depth 1, Right: Split on feature: 1
  -- Left leaf node with indices [8]
  -- Right leaf node with indices [2, 3, 6, 9]
